#### Signal Collection & Explicit Ratings

#### Sparsity Formula
$$\text{Sparsity(Density)} = \left(\frac{N_{\text{ratings}}}{N_{\text{users}} \times N_{\text{movies}}} \right) \times 100$$

Sparsity is the percentage of "empty" or "latent" cells in the matrix. It represents the information gap the Collaborative Filtering (CF) model must overcome.
$$\text{Sparsity} = \left( 1 - \frac{N_{\text{ratings}}}{N_{\text{users}} \times N_{\text{movies}}} \right) \times 100$$

- However, when the matrix is extremely sparse (as is typical in recommendation datasets like MovieLens), the number of actual ratings is very small compared to total possible entries.

- The difference between the two formulas is negligible (on the order of 0.0001–0.001%).

In [2]:
# Load and inspect ratings data for CF

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import pickle
from pathlib import Path

print("=" * 40)
print("SECTION 9: COLLABORATIVE FILTERING - DATA PREPARATION")
print("=" * 40)

# Load clean ratings
ratings = pd.read_csv("../data/processed/ratings_clean.csv")
print(f"\nRating shape: {ratings.shape}")
print(f"Columns: {ratings.columns.tolist()}")


# Basic Statistic
n_users = ratings["userId"].nunique()
n_movies = ratings["tmdbId"].nunique()
n_ratings = len(ratings)

print(f"\nUsers: {n_users:,}")
print(f"\nMovies: {n_movies:,}")
print(f"\nRatings: {n_ratings:,}")
# Sparsity = percentage of missing ratings
# Approximation valid for highly sparse matrices (difference negligible)
# Strict version: sparsity = (1 - (n_ratings / (n_users * n_movies))) * 100
print(f"Sparsity: {n_ratings / (n_users * n_movies) * 100:.4f}%")

SECTION 9: COLLABORATIVE FILTERING - DATA PREPARATION

Rating shape: (25981438, 6)
Columns: ['userId', 'movieId', 'rating', 'timestamp', 'tmdbId', 'year']

Users: 270,882

Movies: 44,702

Ratings: 25,981,438
Sparsity: 0.2146%


- The true sparsity is 99.7854% (meaning 99.7854% of possible user-movie pairs have no rating).
- The approximation 0.2146% is the proportion of filled entries — it is the inverse of the missing proportion.

##### Matrix Interaction & Sparsity Analysis
1. Theoretical Maximum Interactions (Total Search Space):
    - Formula: $N_{\text{users}} \times N_{\text{movies}}$
    - Calculation: $270,888 \times 44,702 \approx 12,109,235,376$ (12.11 Billion)
2. Observed Interactions (Actual Ratings):
    - Metric: $N_{\text{ratings}}$
    - Total: $25,981,438$ (~26 Million)
3. Matrix Density & Sparsity Results:
    - Density (Fill Rate):
    - $$\frac{25,981,438}{12,109,235,376} \times 100 \approx \mathbf{0.2146\%}$$
4. Sparsity (Missing Data):
    - $$100\% - 0.2146\% \approx \mathbf{99.7854\%}$$

#### Interpretation:
With a density of 0.2146%, the dataset exceeds the minimum threshold (0.1%) required for a stable Collaborative Filtering model. The system possesses sufficient interaction "connective tissue" to proceed with SVD++ Matrix Factorization.


### User Mean-Centering (also known as Mean-Normalization)

#### The Formula
$$r_{ui}^* = r_{ui} - \bar{r}_u$$

Where:

- $r_{ui}^*$: The Mean-Centered Rating (the result).
- $r_{ui}$: The Original Rating given by user $u$ to movie $i$ (e.g., 4 stars).
- $\bar{r}_u$: The Average Rating of all movies rated by user $u$.

In [3]:
# User mean-centering - VECTORIZED (remove optimist/pessimist bias)

print("\n" + "=" * 60)
print("STEP 9.2: USER MEAN-CENTERING (VECTORIZED)")
print("=" * 60)

# Compute user means (fast, groupby returns small DataFrame)
user_means = ratings.groupby('userId')['rating'].mean().rename('user_mean')

# Merge means back (vectorized operation)
ratings = ratings.merge(user_means, left_on='userId', right_index=True)

# Vectorized centering (NO apply, NO lambda)
ratings['rating_centered'] = ratings['rating'] - ratings['user_mean']

# Drop the temporary column
ratings = ratings.drop('user_mean', axis=1)

print(f"Centered ratings created for {len(ratings):,} rows")
print("\nCentered rating statistics:")
print(ratings['rating_centered'].describe())


STEP 9.2: USER MEAN-CENTERING (VECTORIZED)
Centered ratings created for 25,981,438 rows

Centered rating statistics:
count    2.598144e+07
mean    -3.199727e-18
std      9.529301e-01
min     -4.490079e+00
25%     -5.333333e-01
50%      9.375000e-02
75%      6.400990e-01
max      4.285714e+00
Name: rating_centered, dtype: float64


In [4]:
# Double-centering - VECTORIZED

print("\n" + "=" * 60)
print("DOUBLE-CENTERING (VECTORIZED)")
print("=" * 60)

# Compute item means of centered ratings
item_means = ratings.groupby('tmdbId')['rating_centered'].mean().rename('item_mean')

# Merge and vectorized operation
ratings = ratings.merge(item_means, left_on='tmdbId', right_index=True)
ratings['rating_double_centered'] = ratings['rating_centered'] - ratings['item_mean']
ratings = ratings.drop('item_mean', axis=1)

print("Double-centered ratings statistics:")
print(ratings['rating_double_centered'].describe())


DOUBLE-CENTERING (VECTORIZED)
Double-centered ratings statistics:
count    2.598144e+07
mean    -1.801583e-17
std      8.722091e-01
min     -4.981469e+00
25%     -4.806821e-01
50%      6.404592e-02
75%      5.663130e-01
max      5.248576e+00
Name: rating_double_centered, dtype: float64


In [5]:
# SAVE CENTERED RATINGS FOR CF TRAINING

print("\n" + "=" * 60)
print("SAVING PROCESSED RATINGS FOR CF")
print("=" * 60)

# Save full DataFrame with all columns
output_path = '../data/processed/ratings_centered.csv'
ratings.to_csv(output_path, index=False)
print(f"✓ Saved: {output_path}")
print(f"  Shape: {ratings.shape}")
print(f"  Memory: {ratings.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Also save a lighter version with just what's needed for Surprise
surprise_data = ratings[['userId', 'tmdbId', 'rating_double_centered']].copy()
surprise_data.columns = ['userId', 'tmdbId', 'rating']  # Rename for Surprise

surprise_path = '../data/processed/ratings_for_surprise.csv'
surprise_data.to_csv(surprise_path, index=False)
print(f"\n✓ Saved: {surprise_path} (for Surprise library)")
print(f"  Shape: {surprise_data.shape}")

# Quick verification
print("\n" + "=" * 60)
print("VERIFICATION")
print("=" * 60)
print(f"Original ratings: {len(ratings):,}")
print(f"Centered ratings mean: {ratings['rating_centered'].mean():.10f}")
print(f"Double-centered mean: {ratings['rating_double_centered'].mean():.10f}")
print(f"Sample user (ID {ratings['userId'].iloc[0]}):")
print(f"  Original: {ratings['rating'].iloc[0]}")
print(f"  Centered: {ratings['rating_centered'].iloc[0]:.3f}")
print(f"  Double-centered: {ratings['rating_double_centered'].iloc[0]:.3f}")


SAVING PROCESSED RATINGS FOR CF
✓ Saved: ../data/processed/ratings_centered.csv
  Shape: (25981438, 8)
  Memory: 3270.67 MB

✓ Saved: ../data/processed/ratings_for_surprise.csv (for Surprise library)
  Shape: (25981438, 3)

VERIFICATION
Original ratings: 25,981,438
Centered ratings mean: -0.0000000000
Double-centered mean: -0.0000000000
Sample user (ID 38150):
  Original: 4.0
  Centered: 0.226
  Double-centered: -0.103


In [7]:
# BUILD USER-ITEM MATRIX FOR CF

print("\n" + "=" * 60)
print("STEP 10: CONSTRUCTING USER-ITEM MATRIX FOR CF")
print("=" * 60)

import pickle
from scipy.sparse import csr_matrix, save_npz
from sklearn.preprocessing import LabelEncoder

# Load your centered ratings
ratings = pd.read_csv('../data/processed/ratings_centered.csv')
print(f"Ratings loaded: {len(ratings):,}")


STEP 10: CONSTRUCTING USER-ITEM MATRIX FOR CF
Ratings loaded: 25,981,438


In [8]:

# CREATE USER AND MOVIE MAPPERS

print("\n" + "=" * 60)
print("STEP 10.3: CREATING ID MAPPERS")
print("=" * 60)

# Create user mapper (userId → index)
unique_users = sorted(ratings['userId'].unique())
user_mapper = {uid: idx for idx, uid in enumerate(unique_users)}
user_inv_mapper = {idx: uid for uid, idx in user_mapper.items()}

print(f"Users: {len(user_mapper):,}")

# Create movie mapper (tmdbId → index) - USE CBF MAPPINGS TO BE CONSISTENT
with open("../models/cbf/movie_id_to_idx.pkl", "rb") as f:
    movie_id_to_idx = pickle.load(f)

with open("../models/cbf/idx_to_movie_id.pkl", "rb") as f:
    idx_to_movie_id = pickle.load(f)

print(f"Movies from CBF: {len(movie_id_to_idx):,}")


STEP 10.3: CREATING ID MAPPERS
Users: 270,882
Movies from CBF: 45,423


In [9]:
# MAP RATINGS TO MATRIX INDICES

print("\n" + "=" * 60)
print("STEP 10.2: MAPPING RATINGS TO INDICES")
print("=" * 60)

# Map users and movies to indices
ratings['user_idx'] = ratings['userId'].map(user_mapper)
ratings['movie_idx'] = ratings['tmdbId'].map(movie_id_to_idx)


# Verify no missing mappings
missing_users = ratings['user_idx'].isna().sum()
missing_movies = ratings['movie_idx'].isna().sum()

print(f"Missing user mappings: {missing_users:,}")
print(f"Missing movie mappings: {missing_movies:,}")

if missing_movies > 0:
    print("\nWARNING: Some movies in ratings are not in CBF index!")
    print("These movies will be excluded from CF matrix.")
    # Filter them out
    ratings = ratings.dropna(subset=['movie_idx']).copy()
    print(f"Ratings after filtering: {len(ratings):,}")


STEP 10.2: MAPPING RATINGS TO INDICES
Missing user mappings: 0
Missing movie mappings: 1

These movies will be excluded from CF matrix.
Ratings after filtering: 25,981,437


In [10]:

# BUILD SPARSE MATRIX

print("\n" + "=" * 60)
print("STEP 10.2: BUILDING SPARSE MATRIX")
print("=" * 60)

# Use double-centered ratings for the matrix
R = csr_matrix(
    (
        ratings['rating_double_centered'].astype(np.float32),
        (ratings['user_idx'].astype(int), ratings['movie_idx'].astype(int))
    ),
    shape=(len(user_mapper), len(movie_id_to_idx)),
    dtype=np.float32
)

print(f"Sparse matrix shape: {R.shape}")
print(f"Non-zero entries: {R.nnz:,}")
print(f"Density: {R.nnz / (R.shape[0] * R.shape[1]) * 100:.6f}%")


STEP 10.2: BUILDING SPARSE MATRIX
Sparse matrix shape: (270882, 45423)
Non-zero entries: 25,981,437
Density: 0.211158%


In [11]:

# SAVE MAPPERS AND MATRIX

print("\n" + "=" * 60)
print("STEP 10.3: SAVING CF ARTIFACTS")
print("=" * 60)

# Save mappers
with open('../models/cf/user_mapper.pkl', 'wb') as f:
    pickle.dump(user_mapper, f)
print("✓ Saved: user_mapper.pkl")

with open('../models/cf/user_inv_mapper.pkl', 'wb') as f:
    pickle.dump(user_inv_mapper, f)
print("✓ Saved: user_inv_mapper.pkl")

with open('../models/cf/movie_mapper.pkl', 'wb') as f:
    pickle.dump(movie_id_to_idx, f)  # Reuse CBF mapper
print("✓ Saved: movie_mapper.pkl")

with open('../models/cf/movie_inv_mapper.pkl', 'wb') as f:
    pickle.dump(idx_to_movie_id, f)  # Reuse CBF mapper
print("✓ Saved: movie_inv_mapper.pkl")

# Save the matrix
save_npz("../data/processed/interactions_matrix.npz", R)
print("✓ Saved: interactions_matrix.npz")


STEP 10.3: SAVING CF ARTIFACTS
✓ Saved: user_mapper.pkl
✓ Saved: user_inv_mapper.pkl
✓ Saved: movie_mapper.pkl
✓ Saved: movie_inv_mapper.pkl
✓ Saved: interactions_matrix.npz


In [12]:
# SAVE CLEAN INTERACTIONS CSV

print("\n" + "=" * 60)
print("STEP 10.1: SAVING INTERACTIONS CSV")
print("=" * 60)

# Create interactions DataFrame (as per spec)
interactions = ratings[['userId', 'tmdbId', 'rating_double_centered']].copy()
interactions.columns = ['userId', 'tmdbId', 'value']
interactions['type'] = 'explicit'  # All are explicit ratings for now

interactions.to_csv('../data/processed/interactions_clean.csv', index=False)
print(f"✓ Saved: interactions_clean.csv ({len(interactions):,} interactions)")


STEP 10.1: SAVING INTERACTIONS CSV
✓ Saved: interactions_clean.csv (25,981,437 interactions)


In [13]:
# FINAL VERIFICATION

print("\n" + "=" * 60)
print("FINAL CF DATA VERIFICATION")
print("=" * 60)

print(f"Users in matrix: {R.shape[0]:,}")
print(f"Movies in matrix: {R.shape[1]:,}")
print(f"Total interactions: {R.nnz:,}")
print(f"Matrix sparsity: {100 - (R.nnz / (R.shape[0] * R.shape[1]) * 100):.4f}% empty")

# Sample check
sample_user = ratings['userId'].iloc[0]
sample_user_idx = user_mapper[sample_user]
sample_movie = ratings['tmdbId'].iloc[0]
sample_movie_idx = movie_id_to_idx[sample_movie]
sample_value = R[sample_user_idx, sample_movie_idx]

print(f"\nSample verification:")
print(f"  User {sample_user} → index {sample_user_idx}")
print(f"  Movie {sample_movie} → index {sample_movie_idx}")
print(f"  Matrix value: {sample_value:.3f}")
print(f"  Original double-centered: {ratings['rating_double_centered'].iloc[0]:.3f}")


FINAL CF DATA VERIFICATION
Users in matrix: 270,882
Movies in matrix: 45,423
Total interactions: 25,981,437
Matrix sparsity: 99.7888% empty

Sample verification:
  User 38150 → index 38149
  Movie 1600 → index 1136
  Matrix value: -0.103
  Original double-centered: -0.103


In [14]:
# IMPLICIT FEEDBACK SIGNALS (SECTION 9.3)

print("\n" + "=" * 60)
print("STEP 9.3: IMPLICIT FEEDBACK SIGNALS")
print("=" * 60)

# Binary implicit: 1 if positive interaction (rating >= 3.5)
ratings['implicit_binary'] = (ratings['rating'] >= 3.5).astype(int)

# Confidence-weighted implicit: scale to 0.1–1.0
ratings['implicit_confidence'] = ratings['rating'] / 5.0

print("Implicit signals added.")
print(ratings[['userId', 'tmdbId', 'rating', 'rating_double_centered', 'implicit_binary', 'implicit_confidence']].head(10))

# Final save (updated interactions with implicit signals)
interactions = ratings[['userId', 'tmdbId', 'rating_double_centered', 'implicit_binary', 'implicit_confidence']].copy()
interactions.columns = ['userId', 'tmdbId', 'explicit_value', 'implicit_binary', 'implicit_confidence']
interactions.to_csv('../data/processed/interactions_clean.csv', index=False)
print("✓ Updated interactions_clean.csv saved with implicit signals")


STEP 9.3: IMPLICIT FEEDBACK SIGNALS
Implicit signals added.
   userId  tmdbId  rating  rating_double_centered  implicit_binary  \
0   38150    1600     4.0               -0.103286                1   
1   44717    8012     3.0               -0.461146                0   
2   44717     807     5.0                1.115038                1   
3   44717     623     3.0               -0.701052                0   
4   45491    8844     4.0                0.549650                1   
5  126622     807     5.0                0.389602                1   
6  126622     629     5.0                0.201711                1   
7  126622   97406     4.0                0.095431                1   
8  126622   11010     5.0                0.448255                1   
9   45491   11860     5.0                1.408991                1   

   implicit_confidence  
0                  0.8  
1                  0.6  
2                  1.0  
3                  0.6  
4                  0.8  
5                 